In [ ]:
import json
import os
import random
import sys
from pathlib import Path

# Giảm phân mảnh VRAM – đặt TRƯỚC khi import torch
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    TrainerControl,
    TrainerState,
)



# ─────────────────────────────────────────────
# Cấu hình
# ─────────────────────────────────────────────
MODEL_NAME  = "google/byt5-small"

# Các file chunks đã được build sẵn bởi scripts/build_chunks.py
TRAIN_CHUNKS = Path("/kaggle/input/datasets/thnhtrung5/data36set/train_chunks.csv")
VAL_CHUNKS   = Path("/kaggle/input/datasets/thnhtrung5/data36set/val_chunks.csv")
TEST_CHUNKS  = Path("/kaggle/input/datasets/thnhtrung5/data36set/test_chunks.csv")

OUTPUT_DIR  = Path("byt5_correction")
CACHE_DIR   = Path("byt5_correction/.tok_cache")

MAX_INPUT_LEN  = 512   # bytes (ByT5 limit) — phải khớp với CHUNK_SIZE trong build_chunks.py
MAX_TARGET_LEN = 512
CHUNK_SIZE     = 512   # phải khớp với CHUNK_SIZE trong build_chunks.py
OVERLAP        = 80    # phải khớp với OVERLAP trong build_chunks.py

# ── Điều chỉnh theo phần cứng ──────────────────────────────────
# | Môi trường         | BATCH | GRAD_ACCUM | eff_batch | Ghi chú             |
# |--------------------|-------|------------|-----------|---------------------|
# | Colab T4 (1 GPU)   |   2   |     8      |    16     |                     |
# | Kaggle T4×2 (2GPU) |   2   |     8      |    32     | ← KHUYẾN NGHỊ      |
# | Kaggle P100 (1GPU) |   2   |     8      |    16     | USE_BF16=False      |
MAX_TRAIN_SAMPLES = 15_000  # dùng hết toàn bộ train
MAX_VAL_SAMPLES   = 500      # tăng lên để eval chính xác hơn
BATCH_SIZE        = 4
EVAL_BATCH_SIZE   = 4
GRAD_ACCUM        = 8        # effective batch = 4 × 8 × 2GPU = 64
EPOCHS            = 5       # tăng từ 2 lên 10
LR                = 3e-4     # tăng lr một chút
WARMUP_RATIO      = 0.06
SEED              = 42

# ── Eval mode ────────────────────────────────────────────────────
# False → dùng cross-entropy loss thật từ logits (nhanh, ít VRAM)
# True  → generate() để đánh giá chất lượng inference thực tế
USE_GENERATE_EVAL = True

# eval_accumulation_steps: sau bao nhiêu bước eval thì flush logits về CPU
# Đặt = 1 để KHÔNG tích luỹ logit 3D trong RAM → tránh OOM khi pad & concat
EVAL_ACCUM_STEPS  = 1

# T4 hỗ trợ bf16 tốt hơn fp16 cho ByT5 (tránh underflow gradient)
# Kaggle T4×2: True | Kaggle P100: False (P100 không hỗ trợ bf16)
USE_BF16 = True


# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def load_split(csv_path: Path, max_samples: int | None = None, split_name: str = "") -> pd.DataFrame:
    """Đọc file chunks CSV (input=noisy, target=clean)."""
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Khong tim thay: {csv_path}\n"
            f"Hay chay truoc: python scripts/build_chunks.py"
        )
    df = pd.read_csv(csv_path)

    if max_samples and len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=SEED).reset_index(drop=True)
        print(f"[INFO] {split_name}: sampled {max_samples:,} / {len(df)+max_samples:,} chunks")
    else:
        print(f"[INFO] {split_name}: {len(df):,} chunks")

    return df


def preprocess_fn(examples, tokenizer):
    """Tokenize input/target cho ByT5."""
    model_inputs = tokenizer(
        text=examples["input"],
        text_target=examples["target"],
        max_length=MAX_INPUT_LEN,
        max_target_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    return model_inputs


class EpochLogCallback(TrainerCallback):
    """In exact match và CER sau mỗi epoch."""

    def __init__(self):
        self._train_acc_sum   = 0.0
        self._train_acc_steps = 0

    # Tích luỹ accuracy trên từng batch train
    def on_prediction_step(self, args, state: TrainerState, control: TrainerControl,
                           **kwargs):
        # Hook này chỉ gọi trong evaluate(), không dùng ở đây
        pass

    def on_step_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        # Không có logits ở hook này – dùng on_log thay thế
        pass

    def on_log(self, args, state: TrainerState, control: TrainerControl,
               logs=None, **kwargs):
        if logs is None:
            return
        # HuggingFace Trainer không tự log accuracy – ta tính ở compute_metrics
        # Hook này chỉ dùng để forward các giá trị đã có
        pass

    def on_evaluate(self, args, state: TrainerState, control: TrainerControl,
                    metrics=None, **kwargs):
        """Gọi sau mỗi lần evaluate (= sau mỗi epoch)."""
        if metrics is None:
            return
        epoch = int(state.epoch) if state.epoch else "?"
        val_loss = metrics.get("eval_loss", None)
        val_exact_match = metrics.get("eval_exact_match", None)
        val_cer = metrics.get("eval_cer", None)
        print(f"\n[EPOCH {epoch}] ", end="")
        if val_loss  is not None:
            print(f"val_loss={val_loss:.4f}  ", end="")
        if val_exact_match is not None:
            print(f"val_exact_match={val_exact_match*100:.2f}%  ", end="")
        if val_cer is not None:
            print(f"val_cer={val_cer:.4f}", end="")
        print()


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main() -> None:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[INFO] Device: {device}")

    # ── 1. Load 3 splits từ file đã build sẵn ─────────────────────
    print("\n[INFO] Loading pre-built chunk datasets...")
    train_df = load_split(TRAIN_CHUNKS, max_samples=MAX_TRAIN_SAMPLES, split_name="Train")
    val_df   = load_split(VAL_CHUNKS,   max_samples=MAX_VAL_SAMPLES,   split_name="Val  ")
    test_df  = load_split(TEST_CHUNKS,  max_samples=None,              split_name="Test ")

    print(f"\n[INFO] Sizes → Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

    # ── 2. Chuyển sang HuggingFace Dataset ────────────────────────
    train_ds = Dataset.from_pandas(train_df[["input", "target"]], preserve_index=False)
    val_ds   = Dataset.from_pandas(val_df[["input", "target"]],   preserve_index=False)
    test_ds  = Dataset.from_pandas(test_df[["input", "target"]],  preserve_index=False)

    # ── 3. Load tokenizer & model ─────────────────────────────────
    print(f"\n[INFO] Loading tokenizer & model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        tie_word_embeddings=False,
    )
    model.to(device)
    # model.config.decoder_start_token_id = tokenizer.pad_token_id
    # model.config.forced_bos_token_id = None
    
    # model.generation_config.decoder_start_token_id = tokenizer.pad_token_id
    # model.generation_config.forced_bos_token_id = None
    # print("decoder_start_token_id (config)     :", model.config.decoder_start_token_id)
    # print("decoder_start_token_id (gen_config) :", model.generation_config.decoder_start_token_id)
    # print("forced_bos_token_id (config)        :", model.config.forced_bos_token_id)
    # print("forced_bos_token_id (gen_config)    :", model.generation_config.forced_bos_token_id)
    # print("pad_token_id                        :", tokenizer.pad_token_id)
    # print("bos_token_id                        :", tokenizer.bos_token_id)

    # ── 4. Tokenize (có cache) ─────────────────────────────────────
    print("[INFO] Tokenizing datasets...")
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    tok_fn = lambda ex: preprocess_fn(ex, tokenizer)

    train_tok = train_ds.map(
        tok_fn, batched=True,
        remove_columns=["input", "target"],
        cache_file_name=str(CACHE_DIR / f"train_{len(train_df)}.arrow"),
    )
    val_tok = val_ds.map(
        tok_fn, batched=True,
        remove_columns=["input", "target"],
        cache_file_name=str(CACHE_DIR / f"val_{len(val_df)}.arrow"),
    )
    test_tok = test_ds.map(
        tok_fn, batched=True,
        remove_columns=["input", "target"],
        cache_file_name=str(CACHE_DIR / f"test_{len(test_df)}.arrow"),
    )

    collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

    # ── 5. Training arguments ──────────────────────────────────────
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


    steps_per_epoch = max(1, len(train_tok) // (BATCH_SIZE * GRAD_ACCUM))
    total_steps     = steps_per_epoch * EPOCHS
    warmup_steps    = max(100, int(total_steps * WARMUP_RATIO))
    eff_batch       = BATCH_SIZE * GRAD_ACCUM
    print(f"\n[INFO] effective_batch={eff_batch} (per_device={BATCH_SIZE} x accum={GRAD_ACCUM})")
    print(f"[INFO] steps/epoch={steps_per_epoch} | total={total_steps} | warmup={warmup_steps}")

    # Chọn precision: bf16 > fp16 cho ByT5 trên T4/A100
    use_bf16 = USE_BF16 and (device == "cuda") and torch.cuda.is_bf16_supported()
    use_fp16 = (not use_bf16) and (device == "cuda")
    if device == "cuda":
        prec = "bf16" if use_bf16 else "fp16"
        print(f"[INFO] Precision: {prec}")

    # ── compute_metrics: tính token accuracy từ logits ────────────
    def compute_metrics(eval_preds):
        """
        eval_preds.predictions : token ids (batch, seq) khi predict_with_generate=True
        eval_preds.label_ids   : (batch, seq) với -100 ở padding

        Lưu ý: khi eval_accumulation_steps=1, Trainer trả về predictions
        dưới dạng list/tuple các chunk có seq_len khác nhau → NumPy không
        thể stack thành array đồng nhất → phải xử lý thủ công.
        """
        predictions, label_ids = eval_preds

        def _to_ids(obj):
            if isinstance(obj, np.ndarray):
                if obj.ndim in {2, 3}:
                    return list(obj)
                return list(obj)
            out = []
            for chunk in obj:
                chunk = np.asarray(chunk)
                if chunk.ndim >= 2:
                    out.extend(chunk)
                else:
                    out.append(chunk)
            return out

        def _pad_labels(labels):
            return [np.where(seq == -100, tokenizer.pad_token_id, seq) for seq in labels]

        def levenshtein(a: str, b: str) -> int:
            if a == b:
                return 0
            n, m = len(a), len(b)
            if n == 0:
                return m
            if m == 0:
                return n
            dp = list(range(m + 1))
            for i in range(1, n + 1):
                prev = dp[0]
                dp[0] = i
                for j in range(1, m + 1):
                    cur = dp[j]
                    if a[i - 1] == b[j - 1]:
                        dp[j] = prev
                    else:
                        dp[j] = min(prev + 1, dp[j] + 1, dp[j - 1] + 1)
                    prev = cur
            return dp[m]

        try:
            pred_list  = _to_ids(predictions)
            label_list = _to_ids(label_ids)
            label_list = _pad_labels(label_list)

            pred_texts = tokenizer.batch_decode(pred_list, skip_special_tokens=True)
            label_texts = tokenizer.batch_decode(label_list, skip_special_tokens=True)

            exact_matches = 0
            total_cer = 0.0
            total = 0
            for pred_text, label_text in zip(pred_texts, label_texts):
                pred_text = pred_text.strip()
                label_text = label_text.strip()
                exact_matches += int(pred_text == label_text)
                dist = levenshtein(label_text, pred_text)
                total_cer += dist / max(1, len(label_text))
                total += 1

            exact_match = exact_matches / total if total > 0 else 0.0
            cer = total_cer / total if total > 0 else 0.0
        except Exception as e:
            print(f"[WARN] compute_metrics error: {e}")
            exact_match = 0.0
            cer = 1.0

        return {
            "exact_match": exact_match,
            "cer": cer,
        }

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,   
        gradient_accumulation_steps=GRAD_ACCUM,
        eval_accumulation_steps=EVAL_ACCUM_STEPS,     
        gradient_checkpointing=True,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        weight_decay=0.02,
        label_smoothing_factor=0.05,   
        predict_with_generate=True,
        generation_max_length=512 if USE_GENERATE_EVAL else None,
        generation_num_beams=1 if USE_GENERATE_EVAL else 1,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        logging_steps=max(10, steps_per_epoch // 5),
        report_to="none",
        seed=SEED,
        bf16=use_bf16,
        fp16=use_fp16,
        dataloader_num_workers=2,
        use_cpu=(device == "cpu"),
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=None,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=5),
        ],
    )

    # ── 6. Train ───────────────────────────────────────────────────
    print("\n[INFO] Starting training...")
    trainer.train()
    # ── 7. Đánh giá trên Test set ──────────────────────────────────
    print("\n[INFO] Evaluating on TEST set...")
    test_results = trainer.evaluate(eval_dataset=test_tok, metric_key_prefix="test")
    print(f"[TEST] test_loss = {test_results.get('test_loss', 'N/A'):.4f}")
    test_exact = test_results.get('test_exact_match', None)
    test_cer = test_results.get('test_cer', None)
    if test_exact is not None:
        print(f"[TEST] test_exact_match = {test_exact*100:.2f}%")
    if test_cer is not None:
        print(f"[TEST] test_cer = {test_cer:.4f}")
    # ── 8. Lưu model ──────────────────────────────────────────────
    print(f"\n[INFO] Saving best model to: {OUTPUT_DIR}/")
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))

    meta = {
        "base_model":     MODEL_NAME,
        "max_input_len":  MAX_INPUT_LEN,
        "max_target_len": MAX_TARGET_LEN,
        "chunk_size":     CHUNK_SIZE,    
        "overlap":        OVERLAP,        
        "train_samples":  len(train_df),
        "val_samples":    len(val_df),
        "test_samples":   len(test_df),
        "test_loss":      test_results.get("test_loss"),
        "epochs_trained": EPOCHS,
    }
    with open(OUTPUT_DIR / "train_meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    print(f"\n{'='*55}")
    print(f"[OK] Training xong! Model luu tai: {OUTPUT_DIR}/")
    print(f"     test_loss     = {test_results.get('test_loss', 'N/A'):.4f}")
    test_exact = test_results.get('test_exact_match', None)
    test_cer = test_results.get('test_cer', None)
    if test_exact is not None:
        print(f"     test_exact_match = {test_exact*100:.2f}%")
    if test_cer is not None:
        print(f"     test_cer = {test_cer:.4f}")
    print(f"\n     Load lai: AutoModelForSeq2SeqLM.from_pretrained('{OUTPUT_DIR}')")
    print(f"     Test thu : python test_byt5.py")

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

MODEL_PATH = "byt5_correction"
TEST_CSV = "/kaggle/input/datasets/thnhtrung5/data36set/test_chunks.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(device)
model.eval()

test_df = pd.read_csv(TEST_CSV)

# Lấy ngẫu nhiên 100 mẫu
test_df = test_df.sample(n=200, random_state=42).reset_index(drop=True)

BATCH_SIZE = 32      
NUM_BEAMS = 4        


def levenshtein(a, b):
    if a == b:
        return 0

    n, m = len(a), len(b)

    if n == 0:
        return m

    if m == 0:
        return n

    dp = list(range(m + 1))

    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i

        for j in range(1, m + 1):
            cur = dp[j]

            if a[i - 1] == b[j - 1]:
                dp[j] = prev
            else:
                dp[j] = min(
                    prev + 1,
                    dp[j] + 1,
                    dp[j - 1] + 1,
                )

            prev = cur

    return dp[m]


exact_matches = 0
total_cer = 0.0

predictions = []

first_char_matches = 0
first_char_total = 0

missing_first_char_count = 0
wrong_first_char_examples = []
missing_first_char_examples = []

inputs_list = test_df["input"].astype(str).tolist()
targets = test_df["target"].astype(str).tolist()
def fix_first_char(pred: str, inp: str) -> str:
    if not pred or not inp:
        return pred

    LEET_MAP = {'0':'o','1':'i','3':'e','4':'a','5':'s','6':'g','7':'t','8':'b','9':'g'}
    inp0 = inp[0]

    # Nếu ký tự đầu là replacement char (?) → model generate byte lỗi
    # → restore từ input
    if pred[0] == '\ufffd':
        first = LEET_MAP.get(inp0, inp0)
        # Giữ case của input
        if inp0.isupper():
            first = first.upper()
        else:
            first = first.lower()
        return first + pred[1:]  # bỏ ? , thay bằng char đúng

    # Nếu pred[0] khớp expected → chỉ fix case nếu cần
    expected = LEET_MAP.get(inp0, inp0)
    if pred[0].lower() == expected.lower():
        if inp0.isupper() and pred[0].islower():
            return pred[0].upper() + pred[1:]
        return pred

    # Nếu bị mất char đầu (pred bắt đầu bằng inp[1:3])
    if len(inp) >= 2 and pred[:2].lower() == inp[1:3].lower():
        first = LEET_MAP.get(inp0, inp0)
        if inp0.isupper():
            first = first.upper()
        return first + pred

    return pred
    
    # Nếu pred KHÔNG bắt đầu bằng expected → prepend
    # Kiểm tra xem pred có phải là target[1:] không (missing first char)
    # Heuristic: nếu inp[1:3] xuất hiện gần đầu pred → bị mất char đầu
    if len(inp) >= 2 and pred[:2].lower() == inp[1:3].lower():
        # Restore first char
        first = LEET_MAP.get(inp0, inp0)
        if inp0.isupper():
            first = first.upper()
        return first + pred
    
    return pred
for start in tqdm(range(0, len(inputs_list), BATCH_SIZE)):

    batch_inputs = inputs_list[start:start+BATCH_SIZE]
    batch_targets = targets[start:start+BATCH_SIZE]

    encoded = tokenizer(
        batch_inputs,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **encoded,
            max_new_tokens=512,
            num_beams=4,
            early_stopping=True,
            decoder_start_token_id=tokenizer.pad_token_id, 
        )

    def safe_decode(token_ids, tokenizer):
        ids = [t for t in token_ids if t not in (0, 1)]
        byte_vals = []
        for t in ids:
            if 3 <= t <= 258:
                byte_vals.append(t - 3)
        raw_bytes = bytes(byte_vals)
        return raw_bytes.decode("utf-8", errors="replace")

    preds = [safe_decode(out.tolist(), tokenizer) for out in outputs]

    for inp, pred, target in zip(batch_inputs, preds, batch_targets):

        pred = pred.strip()
        pred = fix_first_char(pred, inp) 
        target = target.strip()

        predictions.append(pred)

        # Exact Match
        if pred == target:
            exact_matches += 1

        # CER
        dist = levenshtein(target, pred)
        total_cer += dist / max(1, len(target))

        # First Character Accuracy
        if len(pred) > 0 and len(target) > 0:

            first_char_total += 1

            if pred[0] == target[0]:
                first_char_matches += 1
            else:

                if len(wrong_first_char_examples) < 20:
                    wrong_first_char_examples.append({
                        "input": inp,
                        "target": target,
                        "pred": pred
                    })

        # Missing first character check
        if len(target) >= 2:

            if pred == target[1:]:
                missing_first_char_count += 1

                if len(missing_first_char_examples) < 20:
                    missing_first_char_examples.append({
                        "input": inp,
                        "target": target,
                        "pred": pred
                    })

exact_match = exact_matches / len(test_df)
first_char_acc = (
    first_char_matches / first_char_total
    if first_char_total > 0 else 0
)
cer = total_cer / len(test_df)

print("=" * 70)
print(f"Exact Match        : {exact_match:.4f}")
print(f"CER                : {cer:.4f}")
print(f"First Char Accuracy: {first_char_acc:.4f}")
print(f"Missing First Char : {missing_first_char_count}")
print("=" * 70)

print("\nFIRST CHARACTER ERRORS\n")

for ex in wrong_first_char_examples:

    print("INPUT : ", repr(ex["input"][:120]))
    print("TARGET: ", repr(ex["target"][:120]))
    print("PRED  : ", repr(ex["pred"][:120]))
    print()

print("\nMISSING FIRST CHARACTER EXAMPLES\n")

for ex in missing_first_char_examples:

    print("INPUT : ", repr(ex["input"][:120]))
    print("TARGET: ", repr(ex["target"][:120]))
    print("PRED  : ", repr(ex["pred"][:120]))
    print()

In [ ]:
wrong = 0

for inp, pred, target in zip(inputs_list, predictions, targets):
    if pred != target:
        wrong += 1

        print("=" * 100)
        print("INPUT : ", repr(inp))
        print("TARGET:", repr(target))
        print("PRED  :", repr(pred))

        dist = levenshtein(target, pred)
        print("Distance:", dist)
        print("CER:", dist / max(1, len(target)))

        if wrong >= 20:
            break

In [ ]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/datasets/thnhtrung5/data36set/train_chunks.csv")

# Xem raw bytes của input/target đầu tiên
for i in range(5):
    inp = train_df["input"].iloc[i]
    tgt = train_df["target"].iloc[i]
    print(f"=== Sample {i} ===")
    print(f"INPUT  bytes[0:4]: {inp[:4].encode('utf-8').hex()}")
    print(f"TARGET bytes[0:4]: {tgt[:4].encode('utf-8').hex()}")
    print(f"INPUT  : {repr(inp[:80])}")
    print(f"TARGET : {repr(tgt[:80])}")
    print()

In [ ]:
print(outputs[0][:20])

In [ ]:
print(tokenizer.convert_ids_to_tokens(outputs[0][:20]))

In [ ]:
# Debug: xem raw token ids
for i, out in enumerate(outputs[:3]):
    raw = out.tolist()
    print(f"Sample {i} raw ids (first 10): {raw[:10]}")
    print(f"Sample {i} decoded: {repr(tokenizer.decode(out, skip_special_tokens=False)[:50])}")

In [ ]:
# ── INFERENCE TEST VỚI FIX FIRST CHAR ─────────────────────────

LEET_MAP = {'0':'o','1':'i','3':'e','4':'a','5':'s','6':'g','7':'t','8':'b','9':'g'}

def safe_decode(token_ids, tokenizer):
    ids = [t for t in token_ids if t not in (0, 1)]
    byte_vals = [t - 3 for t in ids if 3 <= t <= 258]
    return bytes(byte_vals).decode("utf-8", errors="replace")

def fix_first_char(pred: str, inp: str) -> str:
    if not pred or not inp:
        return pred
    inp0 = inp[0]
    # Ký tự đầu bị corrupt → restore từ input
    if pred[0] == '\ufffd':
        first = LEET_MAP.get(inp0, inp0)
        return (first.upper() if inp0.isupper() else first.lower()) + pred[1:]
    expected = LEET_MAP.get(inp0, inp0)
    # Đúng char, chỉ fix case nếu cần
    if pred[0].lower() == expected.lower():
        if inp0.isupper() and pred[0].islower():
            return pred[0].upper() + pred[1:]
        return pred
    # Bị mất char đầu → prepend
    if len(inp) >= 2 and pred[:2].lower() == inp[1:3].lower():
        first = LEET_MAP.get(inp0, inp0)
        return (first.upper() if inp0.isupper() else first) + pred
    return pred

def levenshtein(a, b):
    if a == b: return 0
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = prev if a[i-1] == b[j-1] else min(prev+1, dp[j]+1, dp[j-1]+1)
            prev = cur
    return dp[m]

# Load model tốt nhất vừa train
model.eval()

# Lấy 200 mẫu test
sample_df = test_df.sample(n=min(200, len(test_df)), random_state=42).reset_index(drop=True)
inputs_list = sample_df["input"].astype(str).tolist()
targets_list = sample_df["target"].astype(str).tolist()

BATCH_SIZE_INFER = 8
exact_matches = 0
total_cer = 0.0
first_char_matches = 0
first_char_total = 0
wrong_examples = []

from tqdm.auto import tqdm

for start in tqdm(range(0, len(inputs_list), BATCH_SIZE_INFER)):
    batch_inputs  = inputs_list[start:start+BATCH_SIZE_INFER]
    batch_targets = targets_list[start:start+BATCH_SIZE_INFER]

    encoded = tokenizer(
        batch_inputs,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **encoded,
            max_new_tokens=512,
            num_beams=4,
            early_stopping=True,
            # decoder_start_token_id=tokenizer.pad_token_id,
        )

    preds = [safe_decode(out.tolist(), tokenizer) for out in outputs]

    for inp, pred, target in zip(batch_inputs, preds, batch_targets):
        pred   = fix_first_char(pred.strip(), inp)
        target = target.strip()

        if pred == target:
            exact_matches += 1

        dist = levenshtein(target, pred)
        total_cer += dist / max(1, len(target))

        if len(pred) > 0 and len(target) > 0:
            first_char_total += 1
            if pred[0] == target[0]:
                first_char_matches += 1
            elif len(wrong_examples) < 10:
                wrong_examples.append({"input": inp, "target": target, "pred": pred})

n = len(sample_df)
print("=" * 60)
print(f"Exact Match        : {exact_matches/n:.4f}")
print(f"CER                : {total_cer/n:.4f}")
print(f"First Char Accuracy: {first_char_matches/first_char_total:.4f}")
print("=" * 60)

print("\nWRONG FIRST CHAR EXAMPLES:")
for ex in wrong_examples:
    print(f"  INPUT : {repr(ex['input'][:80])}")
    print(f"  TARGET: {repr(ex['target'][:80])}")
    print(f"  PRED  : {repr(ex['pred'][:80])}")
    print()

In [ ]:
# ── END-TO-END PIPELINE ────────────────────────────────────────────────────────
# Nhận 1 đoạn văn thô (noisy) → tự chunk → correct → stitch → trả về kết quả
# Không cần doc_id hay chunk_idx, dùng được ngay trên dữ liệu thực

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_PATH = "byt5_correction"
CHUNK_SIZE = 512
OVERLAP    = 80

device    = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(device)
model.eval()

# ── Helpers ───────────────────────────────────────────────────────────────────

LEET_MAP = {'0':'o','1':'i','3':'e','4':'a','5':'s','6':'g','7':'t','8':'b','9':'g'}

def safe_decode(token_ids):
    ids = [t for t in token_ids if t not in (0, 1)]
    byte_vals = [t - 3 for t in ids if 3 <= t <= 258]
    return bytes(byte_vals).decode("utf-8", errors="replace")

def fix_first_char(pred: str, inp: str) -> str:
    if not pred or not inp:
        return pred
    inp0 = inp[0]
    if pred[0] == '\ufffd':
        first = LEET_MAP.get(inp0, inp0)
        return (first.upper() if inp0.isupper() else first.lower()) + pred[1:]
    expected = LEET_MAP.get(inp0, inp0)
    if pred[0].lower() == expected.lower():
        if inp0.isupper() and pred[0].islower():
            return pred[0].upper() + pred[1:]
        return pred
    if len(inp) >= 2 and pred[:2].lower() == inp[1:3].lower():
        first = LEET_MAP.get(inp0, inp0)
        return (first.upper() if inp0.isupper() else first) + pred
    return pred

# ── Chunk ─────────────────────────────────────────────────────────────────────

def make_chunks(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> list[str]:
    """Cắt text thành các chunk theo byte boundary, giống build_chunks.py."""
    chunks = []
    start = 0
    encoded = text.encode("utf-8")
    while start < len(encoded):
        end = start + chunk_size
        chunk_bytes = encoded[start:end]
        # Tránh cắt giữa multibyte UTF-8
        while len(chunk_bytes) > 0:
            try:
                chunk_bytes.decode("utf-8")
                break
            except UnicodeDecodeError:
                chunk_bytes = chunk_bytes[:-1]
        chunks.append(chunk_bytes.decode("utf-8"))
        start += chunk_size - overlap
    return chunks

# ── Correct batch ─────────────────────────────────────────────────────────────

def correct_batch(chunks: list[str], num_beams: int = 4, batch_size: int = 8) -> list[str]:
    preds = []
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=CHUNK_SIZE,
        ).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **encoded,
                max_new_tokens=CHUNK_SIZE,
                num_beams=num_beams,
                early_stopping=True,
            )
        for inp, out in zip(batch, outputs):
            pred = safe_decode(out.tolist()).strip()
            pred = fix_first_char(pred, inp)
            preds.append(pred)
    return preds

# ── Stitch ────────────────────────────────────────────────────────────────────

def stitch(preds: list[str], overlap: int = OVERLAP) -> str:
    """Ghép các chunk predictions lại, dùng overlap để vá phần đầu bị lỗi."""
    if not preds:
        return ""
    if len(preds) == 1:
        return preds[0]
    result = preds[0]
    for pred in preds[1:]:
        body = pred[overlap:] if len(pred) > overlap else ""
        result = result + body
    return result

# ── Pipeline chính ────────────────────────────────────────────────────────────

def correct_text(text: str, num_beams: int = 4) -> str:
    """
    Nhận 1 đoạn văn noisy bất kỳ, trả về văn bản đã sửa.

    Args:
        text     : đoạn văn thô cần sửa
        num_beams: beam search width (4 = cân bằng, 1 = nhanh hơn)

    Returns:
        Văn bản đã được sửa lỗi
    """
    chunks = make_chunks(text)
    if not chunks:
        return text
    preds = correct_batch(chunks, num_beams=num_beams)
    return stitch(preds)

# ── Test ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Dán thẳng đoạn văn dài vào đây
    noisy_text = """
    h3ll0, my n4m3 1s John. 1 c4m3 1n Un1t3d 574t3s w17h0u7 my p4r3n7s.
    1 7h0gh7 7h47 w1ll b3 m0r3 d1ff1cul f0r m3 bu7 n0w 1 r34l1z3 7h47
    3v3ry7h1ng 1s p0ss1bl3 1f y0u w0rk h4rd 3n0ugh.
    """

    print("INPUT:")
    print(noisy_text.strip())
    print("\nOUTPUT:")
    result = correct_text(noisy_text.strip())
    print(result)

    # Interactive loop
    print("\n" + "="*60)
    print("Nhập đoạn văn để test (Enter 2 lần để chạy, 'quit' để thoát):")
    print("="*60)
    while True:
        lines = []
        while True:
            try:
                line = input()
            except (EOFError, KeyboardInterrupt):
                line = "quit"
            if line.lower() == "quit":
                break
            if line == "" and lines:
                break
            lines.append(line)
        if not lines or lines == ["quit"]:
            print("Thoát.")
            break
        text = "\n".join(lines)
        print("\nOUTPUT:")
        print(correct_text(text))
        print()

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


INPUT:
h3ll0, my n4m3 1s John. 1 c4m3 1n Un1t3d 574t3s w17h0u7 my p4r3n7s.
    1 7h0gh7 7h47 w1ll b3 m0r3 d1ff1cul f0r m3 bu7 n0w 1 r34l1z3 7h47
    3v3ry7h1ng 1s p0ss1bl3 1f y0u w0rk h4rd 3n0ugh.

OUTPUT:
hello, my name is John. I came in United States without my parents.

    I thoght that will be more difficul for me but now i realize that

   everything is possible if you work hard enough.

Nhập đoạn văn để test (Enter 2 lần để chạy, 'quit' để thoát):

OUTPUT:
Dear Principal, I think students should be allowed to bring phones.


OUTPUT:
